
# Infer using MLflow-based Roboflow model

**Author:** Juliana Thomasia C. Marcos\
**Input:** Relevant information for MLflow model retrieval, and Databricks folder path for images to perform inference on\
**Output:** Model predictions and annotated images stored in a Databricks volume

## Overview

This notebook performs inference on new images using a previously registered Roboflow model retrieved from the Databricks MLflow Model Registry. It identifies the appropriate model based on the specified crop, trait, and model type, and loads the version assigned the `champion` alias. It also creates a structured directory in a Databricks volume where users can upload the images to be processed.

Once the images are available, the notebook runs inference using the retrieved model and saves the predictions in COCO format. It also generates annotated images to facilitate visual inspection of the model outputs. The inference results are organized in Databricks according to the crop, trait, contributor, inference identifier, and execution date, allowing users to access and review predictions from individual inference runs.


# Packages import and installation

In [0]:
%pip install ..

In [0]:
dbutils.library.restartPython()

In [0]:
import mlflow
import os
import requests
import zipfile
import subprocess
import time
import pandas as pd
import shutil

from datetime import datetime
from mlflow.models import infer_signature

from ag_vision.constants import paths
from ag_vision.data_io.mlflow_io import roboflow_to_coco, write_coco_output_to_catalog, annotate_images, existing_version_with_alias, run_registered_model_inference

# Variables set up

In [0]:
mlflow.set_registry_uri("databricks-uc")

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")

In [0]:
MODELS_SCHEMA = "cv_models_registry"
CV_MODELS_SCHEMA_NAME = f"{catalog_name}.{MODELS_SCHEMA}"
INFERENCE_SCHEMA_NAME = CV_MODELS_SCHEMA_NAME.replace(MODELS_SCHEMA, "inference")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {INFERENCE_SCHEMA_NAME}")

In [0]:
parameters = dbutils.widgets.getAll()

In [0]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS {INFERENCE_SCHEMA_NAME}.{parameters['crop_type']}")

In [0]:
alias = "champion"
reg_model_name = f"{parameters['trait']}__{parameters['crop_type']}__{parameters['model_type']}"
reg_full_model_name = f"{CV_MODELS_SCHEMA_NAME}.{reg_model_name}"
print(reg_full_model_name)

In [0]:
# [TO Do]: Add relevant values for these parameters to tag during the registration of the model
parameters['model_url'] = reg_full_model_name
parameters['version'] = ""

# Verify availability on Databricks of the model specified

In [0]:
existing = existing_version_with_alias(reg_full_model_name)
model_retrieved = None

if existing:
    print("A relevant model was found and retrieved. Predictions can be performed next.")
    model_retrieved = mlflow.pyfunc.load_model(f"models:/{reg_full_model_name}@champion")
else:
    print(f"No model found for the combination of {parameters['crop_type']} crop and {parameters['trait']} trait. Kindly specify relevant information for an existing model.")

# Upload images on Databricks for inference

In [0]:
date = f'{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}'
samples_path = f"/Volumes/{INFERENCE_SCHEMA_NAME}/{parameters['crop_type']}/{parameters['trait']}/{parameters['contributor']}/{parameters['inference_alias']}/{date}/images"

samples_path = samples_path.replace(".", "/")
if not os.path.exists(samples_path):
    os.makedirs(samples_path)

In [0]:
if samples_path:
    paths.show_volume_folder_link(path=samples_path,
                                  url_name="here",
                                  prefix=f"The images to perform predictions on can be uploaded ",
                                  final_index=True)

In [0]:
samples_df = None

if samples_path:
    samples = os.listdir(samples_path)
    if samples:
        samples_df = pd.DataFrame({"image": [f"{samples_path}/{image_name}" for image_name in samples]})
    else:
        paths.show_volume_folder_link(path=samples_path,
                                    url_name="here",
                                    prefix=f"No images found. Add the images to perform inference on ",
                                    final_index=True)

# Inference based on retrieved model

In [0]:
if samples_df is not None:
        if model_retrieved:
            # Performing inference with the model retrieved from the models registry
            predictions_path = run_registered_model_inference(model_retrieved=model_retrieved,
                                                              samples_df=samples_df,
                                                              samples_path=samples_path,
                                                              parameters=parameters,
                                                              roboflow_to_coco=roboflow_to_coco,
                                                              write_coco_output_to_catalog=write_coco_output_to_catalog,
            )

            if predictions_path:                
                annotate_images(samples_path,
                                predictions_path,
                                parameters["model_type"],
                                existing.tags['roboflow_model_id'])
                
                paths.show_volume_folder_link(path=predictions_path,
                                              url_name="here",
                                              prefix=f"All predictions for the images uploaded are aavailable ")

        else:
                print(f"No model was found in the models registry for the information provided i.e., for crop trait={parameters['crop_type']}, trait={parameters['crop_trait']}, task_type={parameters['model_type']}")
                print("Retry with the correct/closest information to select the appropriate model.")
else:
        print("No image information were found in the Dataframe to perform inference on.")